In [7]:
import math

# Parameters (adjust as needed)
diagonal = 15.00+2*0.05*2/math.sqrt(3)+0.001  # Diagonal distance across the hexagon (mm)
apothem = (diagonal) / 2 * math.sqrt(3) / 2.0  # Apothem (distance from center to flat side)
side_length = 2*apothem  # Side length
a1_x = side_length  # Primitive vector 1 x-component
a1_y = 0.0  # Primitive vector 1 y-component
a2_x = side_length / 2.0
a2_y = side_length * math.sqrt(3.0) / 2.0

min_N = 13  # Start from this ring (note: example code has 13, but comment mentions ring 1; adjust if typo)
max_N = 13 + (3 - 1)  # End at this ring (note: example code ends at 15, but comment mentions ring 6; adjust if needed)

# Open file for writing coordinates (x y z in mm, without units)
with open("coordinates.txt", "w") as coord_file:
    for n1 in range(-max_N, max_N + 1):
        for n2 in range(max(-max_N, -n1 - max_N), min(max_N, -n1 + max_N) + 1):
            # Calculate the "ring" distance from origin using the max norm
            ring = max(abs(n1), abs(n2), abs(n1 + n2))
            if ring >= min_N and ring <= max_N:
                # Calculate position using primitive vectors
                x = n1 * a1_x + n2 * a2_x
                y = n1 * a1_y + n2 * a2_y
                z = 0.0  # Adjust if prisms are offset along z
                # Write to file
                coord_file.write(f"{x} {y} {z}\n")

In [13]:
r_start=15
thickness=2
crystal_xysize=2 #mm
crystal_zsize=10 #mm
thickness_z=2
points = set()


for i in range(r_start,r_start+thickness):
    r=i
    for z in range(0,thickness_z):
        x = 0
        y = r
        d = 1 - r
        while y >= x:
            if z==0:
                points.add((x, y,z))
                points.add((-x, y,z))
                points.add((x, -y,z))
                points.add((-x, -y,z))
                points.add((y, x,z))
                points.add((-y, x,z))
                points.add((y, -x,z))
                points.add((-y, -x,z))
            else:
                points.add((x, y,z))
                points.add((-x, y,z))
                points.add((x, -y,z))
                points.add((-x, -y,z))
                points.add((y, x,z))
                points.add((-y, x,z))
                points.add((y, -x,z))
                points.add((-y, -x,z))

                points.add((x, y,-z))
                points.add((-x, y,-z))
                points.add((x, -y,-z))
                points.add((-x, -y,-z))
                points.add((y, x,-z))
                points.add((-y, x,-z))
                points.add((y, -x,-z))
                points.add((-y, -x,-z))

            x += 1
            if d <= 0:
                d += 2 * x + 1
            else:
                y -= 1
                d += 2 * (x - y) + 1

with open("coordinates.txt", "w") as coord_file:
    for point in points:
        print(f"{point[0]*crystal_xysize} {point[1]*crystal_xysize} {point[2]*crystal_zsize/2}", file=coord_file)

In [55]:
crystal_xysize = 4+2*0.05#+0.01  # mm
r_start = int(192/crystal_xysize)
thickness =int(35/crystal_xysize)
crystal_zsize = 20+2*0.05  # mm
thickness_z = 8

points = set()

# Calculate the inner and outer radius bounds
r_inner = r_start
r_outer = r_start + thickness-1

# Iterate over a bounding box large enough to cover the outer radius
max_r = r_outer + 1  # Add a bit of margin for discrete points
for x in range(-max_r, max_r + 1):
    for y in range(-max_r, max_r + 1):
        if x == 0 and y == 0:
            continue  # Skip center if not wanted
        r = (x**2 + y**2)**0.5
        if r_inner <= r < r_outer:
            # Add points for each z layer
            for z in range(-thickness_z + 1, thickness_z):  # Symmetric around 0
                points.add((x, y, z))

# Note: This assumes you want points at integer z from -(thickness_z-1) to (thickness_z-1),
# but adjust based on exact z requirements. Original had z=0 and z=±1 for thickness_z=2.

with open("coordinates.txt", "w") as coord_file:
    for point in sorted(points):  # Sort for consistent order
        scaled_x = point[0] * crystal_xysize
        scaled_y = point[1] * crystal_xysize
        scaled_z = point[2] * (crystal_zsize)  # Original scaling for z
        print(f"{scaled_x} {scaled_y} {scaled_z}", file=coord_file)

In [22]:
r_start

91

In [2]:
int(192/(4+2*0.05))*4

184

# Random Geometry

## Cubic Inside

In [ ]:
crystal_trans = 4 + 2 * 0.05  # mm
crystal_long = 20 + 2 * 0.05  # mm
  # Parameter: number of crystals along each side of the wall; adjust as needed
def Generate_Geometry(filename="coordinates.txt",n_stack = 20):
    S = n_stack * crystal_trans
    half_S = S / 2
    half_long = crystal_long / 2

    points = []

    # +z wall: long axis along +z, rotations 0 0 0
    for i in range(n_stack):
        for j in range(n_stack):
            pos_x = crystal_trans * (i - (n_stack - 1) / 2.0)
            pos_y = crystal_trans * (j - (n_stack - 1) / 2.0)
            pos_z = half_S + half_long
            rot_x = 0
            rot_y = 0
            rot_z = 0
            points.append((pos_x, pos_y, pos_z, rot_x, rot_y, rot_z))

    # -z wall: long axis along -z, rotations 180 0 0
    for i in range(n_stack):
        for j in range(n_stack):
            pos_x = crystal_trans * (i - (n_stack - 1) / 2.0)
            pos_y = crystal_trans * (j - (n_stack - 1) / 2.0)
            pos_z = - (half_S + half_long)
            rot_x = 180
            rot_y = 0
            rot_z = 0
            points.append((pos_x, pos_y, pos_z, rot_x, rot_y, rot_z))

    # +x wall: long axis along +x, rotations 0 90 0
    for i in range(n_stack):
        for j in range(n_stack):
            pos_y = crystal_trans * (i - (n_stack - 1) / 2.0)
            pos_z = crystal_trans * (j - (n_stack - 1) / 2.0)
            pos_x = half_S + half_long
            rot_x = 0
            rot_y = 90
            rot_z = 0
            points.append((pos_x, pos_y, pos_z, rot_x, rot_y, rot_z))

    # -x wall: long axis along -x, rotations 0 -90 0
    for i in range(n_stack):
        for j in range(n_stack):
            pos_y = crystal_trans * (i - (n_stack - 1) / 2.0)
            pos_z = crystal_trans * (j - (n_stack - 1) / 2.0)
            pos_x = - (half_S + half_long)
            rot_x = 0
            rot_y = -90
            rot_z = 0
            points.append((pos_x, pos_y, pos_z, rot_x, rot_y, rot_z))

    # +y wall: long axis along +y, rotations 90 0 0
    for i in range(n_stack):
        for j in range(n_stack):
            pos_x = crystal_trans * (i - (n_stack - 1) / 2.0)
            pos_z = crystal_trans * (j - (n_stack - 1) / 2.0)
            pos_y = half_S + half_long
            rot_x = 90
            rot_y = 0
            rot_z = 0
            points.append((pos_x, pos_y, pos_z, rot_x, rot_y, rot_z))

    # -y wall: long axis along -y, rotations -90 0 0
    for i in range(n_stack):
        for j in range(n_stack):
            pos_x = crystal_trans * (i - (n_stack - 1) / 2.0)
            pos_z = crystal_trans * (j - (n_stack - 1) / 2.0)
            pos_y = - (half_S + half_long)
            rot_x = -90
            rot_y = 0
            rot_z = 0
            points.append((pos_x, pos_y, pos_z, rot_x, rot_y, rot_z))

    with open(filename, "w") as coord_file:
        for point in sorted(points):  # Sort for consistent order
            x, y, z, rx, ry, rz = point
            print(f"{x} {y} {z} {rx} {ry} {rz}", file=coord_file)



In [68]:
import math

crystal_xysize = 4 + 2 * 0.05  # mm
crystal_zsize = 20 + 2 * 0.05  # mm
R = 100  # Parameter: radius to the center of the crystals in mm; adjust as needed

half_z = crystal_zsize / 2
r_inner = R - half_z
if r_inner <= 0:
    print("Radius too small, crystals would overlap at center.")
    # You may need to increase R
else:
    adjustment_factor = (R / r_inner) ** 2
    packing_factor =math.sqrt(4.1**2+4.1**2)  # Increased from 2.0 to ensure min distance > diagonal
    effective_area_per_crystal = crystal_xysize ** 2 * packing_factor * adjustment_factor
    N = int(4 * math.pi * R ** 2 / effective_area_per_crystal + 0.5)

    golden = (1 + math.sqrt(5)) / 2

    points = []
    for i in range(N):
        z_norm = -1 + 2 * (i + 0.5) / N
        theta = math.acos(z_norm)
        phi = 2 * math.pi * i / golden

        pos_x = R * math.sin(theta) * math.cos(phi)
        pos_y = R * math.sin(theta) * math.sin(phi)
        pos_z = R * math.cos(theta)

        rot_x = 0
        rot_y = math.degrees(theta)
        rot_z = math.degrees(phi) % 360

        points.append((pos_x, pos_y, pos_z, rot_x, rot_y, rot_z))

    with open("coordinates.txt", "w") as coord_file:
        for point in sorted(points):
            x, y, z, rx, ry, rz = point
            print(f"{x} {y} {z} {rx} {ry} {rz}", file=coord_file)